In [53]:
# Import Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
import logging
import gwpy
import sys
from gwpy.timeseries import TimeSeries

In [4]:
# Logger
logging.basicConfig(
    level=20, # set to 10 (DEBUG) for Development/ 20 (INFO) for PROD
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True
)

In [5]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Ligo_pipeline") \
    .getOrCreate()

In [6]:
detector = "H1"
gps_start = 1126259462
duration= 4096 # Sec
sample_rate = 4096

In [7]:
# Getting TimeSeries data from gwpy library
data = TimeSeries.fetch_open_data(
    detector, 
    gps_start, 
    gps_start + duration, 
    sample_rate
)

In [9]:
4096 * 4096

16777216

In [10]:
print(data)
print(type(data))
print(len(data))
print(data.sample_rate)
print(data.duration)
print(data.t0)

TimeSeries([5.16251157e-20, 3.72676369e-20, 2.76847613e-20, ...,
            2.65035515e-19, 2.39260773e-19, 2.42696492e-19]
           unit: dimensionless,
           t0: 1126259462.0 s,
           dt: 0.000244140625 s,
           name: Strain,
           channel: None)
<class 'gwpy.timeseries.timeseries.TimeSeries'>
16777216
4096.0 Hz
4096.0 s
1126259462.0 s


In [11]:
# Check for memory
bytes_per_sample = 8 # For LIGO strain float64 = 8 bytes
total_samples = duration * sample_rate # len(data.value)
memory = total_samples * bytes_per_sample
print(f"Memory Usage in MB: {memory/1000000}")
print(f"Memory Usage in MiB (Mebibyte): {memory/1048576}") # It uses powers of 2 --> 2^10=1024, 2^20=1,048,576

Memory Usage in MB: 134.217728
Memory Usage in MiB (Mebibyte): 128.0


In [12]:
print(type(data.value))
print(data.value.shape)
print(data.value[:5])

<class 'numpy.ndarray'>
(16777216,)
[5.16251157e-20 3.72676369e-20 2.76847613e-20 4.03078351e-20
 6.01961406e-20]


In [13]:
# Create bronze record
bronze_metadata = {
    "detector": detector,
    "gps_start": float(data.t0.value),
    "gps_end": float(data.t0.value + data.duration.value),
    "duration": float(data.duration.value),
    "sample_rate": float(data.sample_rate.value),
    "num_samples": len(data.value),
    "source": "GWOSC"
}

In [14]:
bronze_metadata

{'detector': 'H1',
 'gps_start': 1126259462.0,
 'gps_end': 1126263558.0,
 'duration': 4096.0,
 'sample_rate': 4096.0,
 'num_samples': 16777216,
 'source': 'GWOSC'}

In [15]:
bronze_metadata_df = spark.createDataFrame([bronze_metadata])
bronze_metadata_df.printSchema()
bronze_metadata_df.show(truncate=False)

root
 |-- detector: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- gps_end: double (nullable = true)
 |-- gps_start: double (nullable = true)
 |-- num_samples: long (nullable = true)
 |-- sample_rate: double (nullable = true)
 |-- source: string (nullable = true)

+--------+--------+-------------+-------------+-----------+-----------+------+
|detector|duration|gps_end      |gps_start    |num_samples|sample_rate|source|
+--------+--------+-------------+-------------+-----------+-----------+------+
|H1      |4096.0  |1.126263558E9|1.126259462E9|16777216   |4096.0     |GWOSC |
+--------+--------+-------------+-------------+-----------+-----------+------+



In [76]:
window_seconds = 2
samples_per_window = int(sample_rate * window_seconds)
samples_per_window # 16,777,216 / 8192 = 2048 rows/indows

8192

In [17]:
samples_per_window

8192

In [18]:
num_windows = len(data.value) // samples_per_window
print(num_windows)

2048


In [19]:
first_window = data.value[:samples_per_window]
print(type(first_window))
print(first_window.shape)
print(first_window[:5])

<class 'numpy.ndarray'>
(8192,)
[5.16251157e-20 3.72676369e-20 2.76847613e-20 4.03078351e-20
 6.01961406e-20]


In [20]:
# Create Start and End time for the first window
window_id = 0
window_start = float(data.t0.value) + window_id * window_seconds

In [21]:
window_start

1126259462.0

In [42]:
windows = []
t0 = float(data.t0.value)
window_size =  sample_rate * window_seconds
window_numbers = len(data.value) // (window_seconds * sample_rate)
start_idx = 0
window_id = 0
source = "GWOSC"
gps_start = float(data.t0.value)

# For loop is not optimal
for window_counter in range(window_numbers):
    end_idx = start_idx + window_size
    gps_end = float(gps_start) + window_seconds
    windows.append({
        "window_id": window_id,
        "detector": detector,
        "gps_start": gps_start,
        "gps_end": gps_end,
        "window_duration": window_seconds,
        "sample_rate": sample_rate,
        "num_samples": window_size,
        #"raw_signal": data.value[start_idx:end_idx].tolist(),
        "source": source,
        "file_gps_start": float(data.t0.value),
        "file_duration": len(data.value) // sample_rate,
        "start_idx": start_idx,
        "end_idx": end_idx
    })
    
    window_id += 1
    start_idx += window_size
    gps_start = float(gps_start) + window_seconds

In [43]:
bronze_df = spark.createDataFrame(windows)

In [44]:
print(bronze_df.printSchema())
#print(bronze_df.show(truncate=False))
print(bronze_df.count())

root
 |-- detector: string (nullable = true)
 |-- end_idx: long (nullable = true)
 |-- file_duration: long (nullable = true)
 |-- file_gps_start: double (nullable = true)
 |-- gps_end: double (nullable = true)
 |-- gps_start: double (nullable = true)
 |-- num_samples: long (nullable = true)
 |-- sample_rate: long (nullable = true)
 |-- source: string (nullable = true)
 |-- start_idx: long (nullable = true)
 |-- window_duration: long (nullable = true)
 |-- window_id: long (nullable = true)

None
2048


In [45]:
#bronze_df.select(
#    "window_id", "detector", "gps_start", "gps_end", "num_samples"II
#).show(5)

In [46]:
print(len(windows))
#print(len(windows[0]["raw_signal"]))
#print(type(windows[0]["raw_signal"][0]))

2048


In [47]:
bronze_df_small = spark.createDataFrame(windows[:10])

In [48]:
bronze_df_small.count()

10

In [49]:
bronze_df_small.show()

+--------+-------+-------------+--------------+-------------+-------------+-----------+-----------+------+---------+---------------+---------+
|detector|end_idx|file_duration|file_gps_start|      gps_end|    gps_start|num_samples|sample_rate|source|start_idx|window_duration|window_id|
+--------+-------+-------------+--------------+-------------+-------------+-----------+-----------+------+---------+---------------+---------+
|      H1|   8192|         4096| 1.126259462E9|1.126259464E9|1.126259462E9|       8192|       4096| GWOSC|        0|              2|        0|
|      H1|  16384|         4096| 1.126259462E9|1.126259466E9|1.126259464E9|       8192|       4096| GWOSC|     8192|              2|        1|
|      H1|  24576|         4096| 1.126259462E9|1.126259468E9|1.126259466E9|       8192|       4096| GWOSC|    16384|              2|        2|
|      H1|  32768|         4096| 1.126259462E9| 1.12625947E9|1.126259468E9|       8192|       4096| GWOSC|    24576|              2|        3|

In [39]:
row = bronze_df.collect()[1]
gps_start = row["gps_start"]
file_gps_start = row["file_gps_start"]
print(gps_start)
print(file_gps_start)

1126259464.0
1126259462.0


In [78]:
# The optimal way to create Windows/Bronze Dataframe using Spark.range(n)
# The optimal way to create the Windows/Bronze DataFrame using spark.range(n)
t0 = float(data.t0.value)
window_size = sample_rate * window_seconds
window_numbers = len(data.value) // (window_seconds * sample_rate)
source = "GWOSC"

windows_df = spark.range(window_numbers) \
    .withColumnRenamed("id", "window_id") \
    .withColumn("start_idx", col("window_id") * window_size) \
    .withColumn("end_idx", col("start_idx") + window_size) \
    .withColumn("gps_start", lit(t0) + col("window_id") * window_seconds) \
    .withColumn("window_duration", lit(window_seconds)) \
    .withColumn("gps_end", col("gps_start") + col("window_duration")) \
    .withColumn("num_samples", col("end_idx") - col("start_idx")) \
    .withColumn("source", lit(source)) \
    .withColumn("file_gps_start", lit(t0)) \
    .withColumn("file_duration", lit(len(data.value) // sample_rate)) \
    .withColumn("detector", lit(detector))

In [79]:
windows_df.show()

+---------+---------+-------+-------------+---------------+-------------+-----------+------+--------------+-------------+--------+
|window_id|start_idx|end_idx|    gps_start|window_duration|      gps_end|num_samples|source|file_gps_start|file_duration|detector|
+---------+---------+-------+-------------+---------------+-------------+-----------+------+--------------+-------------+--------+
|        0|        0|   8192|1.126259462E9|              2|1.126259464E9|       8192| GWOSC| 1.126259462E9|         4096|      H1|
|        1|     8192|  16384|1.126259464E9|              2|1.126259466E9|       8192| GWOSC| 1.126259462E9|         4096|      H1|
|        2|    16384|  24576|1.126259466E9|              2|1.126259468E9|       8192| GWOSC| 1.126259462E9|         4096|      H1|
|        3|    24576|  32768|1.126259468E9|              2| 1.12625947E9|       8192| GWOSC| 1.126259462E9|         4096|      H1|
|        4|    32768|  40960| 1.12625947E9|              2|1.126259472E9|       819

In [82]:
windows_df.printSchema()

root
 |-- window_id: long (nullable = false)
 |-- start_idx: long (nullable = false)
 |-- end_idx: long (nullable = false)
 |-- gps_start: double (nullable = false)
 |-- window_duration: integer (nullable = false)
 |-- gps_end: double (nullable = false)
 |-- num_samples: long (nullable = false)
 |-- source: string (nullable = false)
 |-- file_gps_start: double (nullable = false)
 |-- file_duration: integer (nullable = false)
 |-- detector: string (nullable = false)



In [85]:
windows_df_parquet = windows_df.write \
    .mode("overwrite") \
    .parquet("../data/bronze/h1_windows")

In [86]:
parquet_df = df.read.parquet("../data/bronze/part-00000-5e46a889-cd4b-44a8-b1ad-a29a7103de0e-c000.snappy.parquet")

NameError: name 'df' is not defined